# From Strawberry to Agentic Text Failures

Three local, educational demonstrations of why an LLM must not be the final authority for exact string validation. Every "tool call" below is a `print()` statement: this notebook does not read files, make network requests beyond downloading the public model, or perform any privileged action.

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` (small enough to make mistakes that are visible in a demo).

The three cases are:

1. Character-counting: a model’s answer disagrees with Python’s exact count.
2. Representation: visually similar strings have different Unicode code points and tokens.
3. Simulated agent boundary: a semantic model judgment disagrees with a deterministic downstream identifier check.

## Setup

Install the two dependencies once in the active Jupyter environment if needed:

```python
%pip install transformers torch
```

The first model run downloads public model weights and can take a moment.

In [1]:
import re
import unicodedata

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def ask_model(prompt, max_new_tokens=24):
    """Deterministic local inference using the model's chat template."""
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print(f'Loaded {MODEL_NAME}')

/private/tmp/agentic-text-lab/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/private/tmp/agentic-text-lab/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded Qwen/Qwen2.5-0.5B-Instruct


## 1. The strawberry problem: approximate reasoning is not an exact string operation

The model is asked the simple human-language question. Python is the control: it produces the actual character count. A disagreement is a model error, but not yet a security incident.

In [2]:
words = ['strawberry', 'terror', 'error', 'arrhythmia', 'raconteur']

for word in words:
    answer = ask_model(
        f'How many lowercase r characters are in {word!r}? Return only the number.'
    )
    print(f'{word:12} | Python: {word.count("r")} | model: {answer!r}')

# In the initial run, this 0.5B model said strawberry had 2 r's,
# terror had 0, and error had 1. Record what your own run says.

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


strawberry   | Python: 3 | model: 'The string "strawberry" contains 2 lowercase letters: s and w. Therefore, there are 2 lowercase'


terror       | Python: 3 | model: 'There is no lowercase letter \'r\' in the word "terror". Therefore, there are 0 lowercase letters in the'


error        | Python: 3 | model: 'There is no lowercase letter \'r\' in the word "error". Therefore, there are 0 lowercase letters in the'


arrhythmia   | Python: 2 | model: 'To determine how many lowercase letters (r) are present in the word "arrhythmia," we need to count each'


raconteur    | Python: 2 | model: 'To determine how many lowercase letters (r) are present in the word "raconteur," we need to count each'


## 2. Same-looking text, different bytes and tokens

These strings resemble `admin` but are not byte-for-byte equal to it. This is not the same failure as character counting: it is a representation mismatch.

In [3]:
samples = {
    'plain': 'admin',
    'cyrillic_i': 'adm\u0456n',
    'dotless_i': 'adm\u0131n',
    'zero_width_space': 'ad\u200bmin',
    'cyrillic_a': '\u0430dmin',
    'trailing_space': 'admin ',
}

for label, value in samples.items():
    ids = tokenizer.encode(value, add_special_tokens=False)
    pieces = [tokenizer.decode([token]) for token in ids]
    print(f'\n[{label}] visible={value!r}')
    print('  code points:', [f'U+{ord(char):04X}' for char in value])
    print('  UTF-8 hex:   ', value.encode('utf-8').hex())
    print('  tokens:      ', ids, pieces)
    print('  exact == admin:', value == 'admin')


[plain] visible='admin'
  code points: ['U+0061', 'U+0064', 'U+006D', 'U+0069', 'U+006E']
  UTF-8 hex:    61646d696e
  tokens:       [2882] ['admin']
  exact == admin: True

[cyrillic_i] visible='admіn'
  code points: ['U+0061', 'U+0064', 'U+006D', 'U+0456', 'U+006E']
  UTF-8 hex:    61646dd1966e
  tokens:       [46929, 26289, 77] ['adm', 'і', 'n']
  exact == admin: False

[dotless_i] visible='admın'
  code points: ['U+0061', 'U+0064', 'U+006D', 'U+0131', 'U+006E']
  UTF-8 hex:    61646dc4b16e
  tokens:       [46929, 16116] ['adm', 'ın']
  exact == admin: False

[zero_width_space] visible='ad\u200bmin'
  code points: ['U+0061', 'U+0064', 'U+200B', 'U+006D', 'U+0069', 'U+006E']
  UTF-8 hex:    6164e2808b6d696e
  tokens:       [329, 15692, 1065] ['ad', '\u200b', 'min']
  exact == admin: False

[cyrillic_a] visible='аdmin'
  code points: ['U+0430', 'U+0064', 'U+006D', 'U+0069', 'U+006E']
  UTF-8 hex:    d0b0646d696e
  tokens:       [1478, 67, 1065] ['а', 'd', 'min']
  exact == admin: Fal

## 3. Simulated agent boundary

Here the model decides whether a request *means* `admin`; the downstream program uses strict equality. The function only prints what a weak workflow **would** attempt. It does not access a file or service.

A mismatch is the interesting result: the model sees a protected name while the deterministic layer sees a different identifier. The correct defense is not a better prompt—it is deterministic validation before any sensitive tool call.

In [4]:
def model_classification(raw):
    return 'PROTECTED' if raw.upper().startswith('PROTECTED') else 'PUBLIC'

def safe_identifier_guard(value):
    # Example defense: only accept known, ASCII identifiers for this endpoint.
    return bool(re.fullmatch(r'[a-z]+', value)) and value in {'guest', 'reader'}

def simulated_agent(filename):
    answer = ask_model(
        'Return exactly PROTECTED or PUBLIC. Treat visually similar Unicode text '
        'as the same word when deciding. Is this filename the protected name admin? '
        f'Filename: {filename!r}',
        max_new_tokens=16,
    )
    decision = model_classification(answer)
    exact_admin = filename == 'admin'
    allowed_by_guard = safe_identifier_guard(filename)

    print(f'input: {filename!r}')
    print(f'model: {answer!r} -> {decision}')
    print(f'downstream exact check: {exact_admin}')
    print(f'conservative guard allows: {allowed_by_guard}')
    if (decision == 'PROTECTED') != exact_admin:
        print('SIMULATED ONLY: model and exact identifier disagree; block the tool call.')
    else:
        print('SIMULATED ONLY: no disagreement in this case.')

for label, value in samples.items():
    print(f'\n--- {label} ---')
    simulated_agent(value)


--- plain ---
input: 'admin'
model: 'PROTECTED' -> PROTECTED
downstream exact check: True
conservative guard allows: False
SIMULATED ONLY: no disagreement in this case.

--- cyrillic_i ---


input: 'admіn'
model: 'PROTECTED' -> PROTECTED
downstream exact check: False
conservative guard allows: False
SIMULATED ONLY: model and exact identifier disagree; block the tool call.

--- dotless_i ---
input: 'admın'
model: 'PROTECTED' -> PROTECTED
downstream exact check: False
conservative guard allows: False
SIMULATED ONLY: model and exact identifier disagree; block the tool call.

--- zero_width_space ---


input: 'ad\u200bmin'
model: 'The filename "admin" is not considered a protected name in the context of Unicode' -> PUBLIC
downstream exact check: False
conservative guard allows: False
SIMULATED ONLY: no disagreement in this case.

--- cyrillic_a ---
input: 'аdmin'
model: 'PROTECTED' -> PROTECTED
downstream exact check: False
conservative guard allows: False
SIMULATED ONLY: model and exact identifier disagree; block the tool call.

--- trailing_space ---


input: 'admin '
model: 'PROTECTED' -> PROTECTED
downstream exact check: False
conservative guard allows: False
SIMULATED ONLY: model and exact identifier disagree; block the tool call.


## Takeaway

The notebook demonstrates two different reasons to keep authorization logic deterministic:

- Natural-language character counting is not a reliable validator.
- Visual or semantic similarity is not identifier equality.

Use the LLM to explain, summarize, or route low-risk work. For authorization, parse and validate data with explicit code, normalize only according to a documented policy, and make the tool layer enforce the same rule independently.